In [ ]:
# 1. Install Python dependencies
!pip install web3 py-solc-x pandas eth-tester

# 2. Install the Solidity Compiler (solc)
# This allows py-solc-x to compile your .sol files locally in Colab
from solcx import install_solc
install_solc('0.8.20')

In [ ]:
import hashlib
import json
from pathlib import Path

import pandas as pd
from solcx import compile_source, install_solc
from web3 import Web3
from web3.providers.eth_tester import EthereumTesterProvider


SOLIDITY_FILE = Path("/content/HallucinationAudit.sol")
CSV_FILE = Path("/content/hallucination_output.csv")
SOLC_VERSION = "0.8.20"
PENALTY_AMOUNT = 10
HASH_SALT = "change-this-to-a-private-secret-salt"


def load_contract_source(solidity_path: Path) -> str:
    if not solidity_path.exists():
        raise FileNotFoundError(f"Solidity file not found: {solidity_path}")
    return solidity_path.read_text(encoding="utf-8")


def compile_contract(source_code: str):
    install_solc(SOLC_VERSION)
    compiled = compile_source(
        source_code,
        solc_version=SOLC_VERSION,
        output_values=["abi", "bin"],
    )
    _, interface = compiled.popitem()
    return interface["abi"], interface["bin"]


def connect_blockchain():
    w3 = Web3(EthereumTesterProvider())
    if not w3.is_connected():
        raise ConnectionError("Failed to connect to local Ethereum tester")
    account = w3.eth.accounts[0]
    w3.eth.default_account = account
    return w3, account


def deploy_contract(w3: Web3, abi, bytecode):
    contract = w3.eth.contract(abi=abi, bytecode=bytecode)
    tx_hash = contract.constructor().transact()
    receipt = w3.eth.wait_for_transaction_receipt(tx_hash)
    deployed = w3.eth.contract(address=receipt.contractAddress, abi=abi)
    return deployed, receipt


def is_hallucination(value) -> bool:
    return str(value).strip().lower() == "yes"


def hash_sensitive_value(value) -> str:
    payload = f"{HASH_SALT}|{str(value).strip()}"
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def read_csv(csv_path: Path) -> pd.DataFrame:
    if not csv_path.exists():
        raise FileNotFoundError(f"CSV file not found: {csv_path}")

    df = pd.read_csv(csv_path)
    required_columns = {
        "round",
        "node",
        "model",
        "answer",
        "ground_truth",
        "Hallucination",
    }
    missing = required_columns - set(df.columns)
    if missing:
        raise ValueError(f"Missing CSV columns: {sorted(missing)}")
    return df


def build_hashed_dataset(df: pd.DataFrame) -> pd.DataFrame:
    hashed_df = pd.DataFrame()
    hashed_df["round"] = df["round"].astype(int)
    hashed_df["hallucination_label"] = df["Hallucination"].astype(str)
    hashed_df["hallucinated"] = df["Hallucination"].apply(is_hallucination)
    hashed_df["node_hash"] = df["node"].apply(hash_sensitive_value)
    hashed_df["model_hash"] = df["model"].apply(hash_sensitive_value)
    hashed_df["answer_hash"] = df["answer"].apply(hash_sensitive_value)
    hashed_df["ground_truth_hash"] = df["ground_truth"].apply(hash_sensitive_value)
    return hashed_df


def build_reputation_scores(hashed_df: pd.DataFrame) -> pd.DataFrame:
    summary_df = (
        hashed_df.groupby("node_hash")
        .agg(
            total_records=("node_hash", "size"),
            hallucination_count=("hallucinated", "sum"),
        )
        .reset_index()
    )
    summary_df["clean_count"] = (
        summary_df["total_records"] - summary_df["hallucination_count"]
    )
    summary_df["reputation_score"] = (
        (summary_df["clean_count"] / summary_df["total_records"]) * 100
    ).round(2)
    return summary_df


def attach_reputation_to_rows(
    hashed_df: pd.DataFrame, reputation_df: pd.DataFrame
) -> pd.DataFrame:
    return hashed_df.merge(
        reputation_df[["node_hash", "reputation_score"]],
        on="node_hash",
        how="left",
    )


def record_csv_on_chain(contract, df: pd.DataFrame, penalty_amount: int):
    tx_receipts = []

    for _, row in df.iterrows():
        tx_hash = contract.functions.recordAudit(
            int(row["round"]),
            str(row["node_hash"]),
            str(row["model_hash"]),
            str(row["answer_hash"]),
            str(row["ground_truth_hash"]),
            bool(row["hallucinated"]),
            int(penalty_amount),
        ).transact()
        receipt = contract.w3.eth.wait_for_transaction_receipt(tx_hash)
        tx_receipts.append(receipt)

    return tx_receipts


def fetch_audit_trail(contract, reputation_map):
    audit_count = contract.functions.getAuditCount().call()
    rows = []

    for index in range(audit_count):
        entry = contract.functions.getAuditEntry(index).call()
        rows.append(
            {
                "audit_index": index,
                "round": entry[0],
                "node_hash": entry[1],
                "model_hash": entry[2],
                "answer_hash": entry[3],
                "ground_truth_hash": entry[4],
                "hallucinated": entry[5],
                "penalty_applied": entry[6],
                "timestamp": entry[7],
                "reputation_score": reputation_map.get(entry[1], 0.0),
            }
        )

    return pd.DataFrame(rows)


def fetch_node_summary(contract, reputation_df: pd.DataFrame):
    summaries = []
    reputation_rows = {
        str(row["node_hash"]): row for _, row in reputation_df.iterrows()
    }

    for node_hash in sorted(reputation_rows):
        total_penalties, hallucination_count, clean_count, exists = (
            contract.functions.getNodeStatus(node_hash).call()
        )
        rep_row = reputation_rows[node_hash]
        summaries.append(
            {
                "node_hash": node_hash,
                "exists": exists,
                "total_records": int(rep_row["total_records"]),
                "total_penalties": total_penalties,
                "hallucination_count": hallucination_count,
                "clean_count": clean_count,
                "reputation_score": float(rep_row["reputation_score"]),
            }
        )
    return pd.DataFrame(summaries)


def sha256_file(file_path: Path) -> str:
    digest = hashlib.sha256()
    with file_path.open("rb") as file_obj:
        for chunk in iter(lambda: file_obj.read(8192), b""):
            digest.update(chunk)
    return digest.hexdigest()


def hash_audit_row(row: pd.Series) -> str:
    row_payload = "|".join(
        [
            str(row["audit_index"]),
            str(row["round"]),
            str(row["node_hash"]),
            str(row["model_hash"]),
            str(row["answer_hash"]),
            str(row["ground_truth_hash"]),
            str(row["hallucinated"]),
            str(row["penalty_applied"]),
            str(row["timestamp"]),
            str(row["reputation_score"]),
        ]
    )
    return hashlib.sha256(row_payload.encode("utf-8")).hexdigest()


def main():
    source_code = load_contract_source(SOLIDITY_FILE)
    abi, bytecode = compile_contract(source_code)
    w3, account = connect_blockchain()
    contract, receipt = deploy_contract(w3, abi, bytecode)

    df = read_csv(CSV_FILE)
    hashed_df = build_hashed_dataset(df)
    reputation_df = build_reputation_scores(hashed_df)
    enriched_df = attach_reputation_to_rows(hashed_df, reputation_df)
    reputation_map = {
        str(row["node_hash"]): float(row["reputation_score"])
        for _, row in reputation_df.iterrows()
    }

    tx_receipts = record_csv_on_chain(contract, enriched_df, PENALTY_AMOUNT)
    audit_df = fetch_audit_trail(contract, reputation_map)
    node_summary_df = fetch_node_summary(contract, reputation_df)
    audit_df["audit_row_hash"] = audit_df.apply(hash_audit_row, axis=1)

    print("Connected account:", account)
    print("Contract address:", contract.address)
    print("Deployment block:", receipt.blockNumber)
    print("Recorded audit transactions:", len(tx_receipts))
    print("\nNode penalty summary:")
    print(node_summary_df.to_string(index=False))
    print("\nAudit trail:")
    print(audit_df.to_string(index=False))

    audit_output_path = Path("/content/blockchain_audit_output.csv")
    node_summary_path = Path("/content/node_penalty_summary.csv")
    reputation_output_path = Path("/content/hallucination_output_hashed_with_reputation.csv")

    audit_df.to_csv(audit_output_path, index=False)
    node_summary_df.to_csv(node_summary_path, index=False)
    enriched_df.to_csv(reputation_output_path, index=False)

    audit_hash = sha256_file(audit_output_path)
    audit_hash_path = Path("/content/blockchain_audit_output.sha256.txt")
    audit_hash_path.write_text(audit_hash, encoding="utf-8")

    metadata = {
        "contract_address": contract.address,
        "deployer_account": account,
        "deployment_block": receipt.blockNumber,
        "penalty_amount": PENALTY_AMOUNT,
        "total_audit_records": int(contract.functions.getAuditCount().call()),
        "audit_csv_sha256": audit_hash,
        "hash_salt_note": "Keep HASH_SALT private and unchanged to reproduce the same hashes.",
    }
    Path("/content/blockchain_run_metadata.json").write_text(
        json.dumps(metadata, indent=2),
        encoding="utf-8",
    )
    print("\nAudit CSV SHA-256:", audit_hash)
    print("\nSaved:")
    print(str(audit_output_path))
    print(str(node_summary_path))
    print(str(reputation_output_path))
    print(str(audit_hash_path))
    print("/content/blockchain_run_metadata.json")


if __name__ == "__main__":
    main()
